In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# LangChain components — fully updated for latest versions
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All imports successful.")

All imports successful.


In [3]:
file_path = os.path.join('..', 'data', 'processed', 'josaa_featured.csv')
df = pd.read_csv(file_path)

print(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Data loaded: 432,524 rows, 23 columns


In [ ]:
# Store your Groq API key
# Get it from https://console.groq.com → API Keys

import os
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY", "")

# Replace 'your_groq_api_key_here' with your actual key
# It looks like: YOUR_GROQ_API_KEY_HERE

print("API key configured.")

API key configured.


In [7]:
# RAG works by converting data into text documents
# Each document represents one meaningful unit of information
# We create documents at the college-program-category-year level
#
# Each document will look like:
# "In 2024, at Indian Institute of Technology Bombay,
#  Computer Science and Engineering program,
#  OPEN category, Gender Neutral, AI quota,
#  the opening rank was 1 and closing rank was 67.
#  This is an IIT, classified as Very High Competition."

def create_document(row):
    """Convert one row of JoSAA data into a text document."""
    text = (
        f"In {row['year']}, at {row['institute']} "
        f"({row['institute_type']}), "
        f"the {row['program']} program "
        f"for {row['seat_type']} category "
        f"({row['gender']} seats, {row['quota']} quota) "
        f"had an opening rank of {row['opening_rank']:,} "
        f"and closing rank of {row['closing_rank']:,}. "
        f"The rank window was {row['rank_window']:,}. "
        f"Competition level: {row['accessibility_label']}. "
        f"Program category: {row['program_category']}."
    )

    metadata = {
        'institute'       : row['institute'],
        'institute_type'  : row['institute_type'],
        'program'         : row['program'],
        'year'            : int(row['year']),
        'seat_type'       : row['seat_type'],
        'category_base'   : row['category_base'],
        'gender_short'    : row['gender_short'],
        'quota'           : row['quota'],
        'opening_rank'    : int(row['opening_rank']),
        'closing_rank'    : int(row['closing_rank']),
        'accessibility_label': row['accessibility_label'],
        'round'           : int(row['round'])
    }

    return Document(page_content=text, metadata=metadata)

print("Document creation function defined.")

Document creation function defined.


In [9]:
# We create documents from final round data only
# to avoid duplicate information across rounds
# Using the last round per institute type per year

last_round = (
    df[df['is_special_round'] == False]
    .groupby(['year', 'institute_type'])['round']
    .max()
    .reset_index()
    .rename(columns={'round': 'last_round'})
)

df_merged = df.merge(last_round, on=['year', 'institute_type'], how='left')

df_rag = df_merged[
    (df_merged['round'] == df_merged['last_round']) &
    (df_merged['is_special_round'] == False) &
    (df_merged['is_pwd'] == False)
].copy()

print(f"Documents to create: {len(df_rag):,}")

# Create documents
# We use a sample for faster indexing
# Full dataset would take too long to embed
print("\nCreating documents...")
documents = []
for _, row in df_rag.iterrows():
    documents.append(create_document(row))

print(f"Documents created: {len(documents):,}")
print(f"\nSample document:")
print(documents[0].page_content)
print(f"\nSample metadata:")
print(documents[0].metadata)

Documents to create: 65,628

Creating documents...
Documents created: 65,628

Sample document:
In 2022, at Assam University, Silchar (GFTI), the Agricultural Engineering (4 Years, Bachelor of Technology) program for EWS category (Female-only (including Supernumerary) seats, HS quota) had an opening rank of 27,646 and closing rank of 27,646. The rank window was 0. Competition level: Moderate Competition. Program category: Other.

Sample metadata:
{'institute': 'Assam University, Silchar', 'institute_type': 'GFTI', 'program': 'Agricultural Engineering (4 Years, Bachelor of Technology)', 'year': 2022, 'seat_type': 'EWS', 'category_base': 'EWS', 'gender_short': 'FO', 'quota': 'HS', 'opening_rank': 27646, 'closing_rank': 27646, 'accessibility_label': 'Moderate Competition', 'round': 6}


In [11]:
# In addition to row-level documents, we create
# summary documents that aggregate trends
# These help answer questions like
# "How has IIT Bombay CSE changed over 5 years?"

summary_docs = []

# Summary 1: College-program-category trend over years
college_program_trend = (
    df_rag.groupby([
        'institute', 'program',
        'category_base', 'gender_short', 'quota'
    ])
    .agg(
        years_available=('year', list),
        closing_ranks=('closing_rank', list),
        min_closing=('closing_rank', 'min'),
        max_closing=('closing_rank', 'max'),
        avg_closing=('closing_rank', 'mean'),
        trend_direction=('closing_rank', lambda x:
            'increasing' if x.iloc[-1] > x.iloc[0]
            else 'decreasing'
            if len(x) > 1 else 'stable')
    )
    .reset_index()
)

for _, row in college_program_trend.iterrows():
    years = sorted(
        zip(row['years_available'], row['closing_ranks']),
        key=lambda x: x[0]
    )
    year_rank_str = ', '.join([
        f"{yr}: {rk:,}" for yr, rk in years
    ])

    text = (
        f"Trend summary for {row['institute']} — "
        f"{row['program']}, "
        f"{row['category_base']} category, "
        f"{row['gender_short']} gender, "
        f"{row['quota']} quota: "
        f"Closing ranks over the years — {year_rank_str}. "
        f"The trend is {row['trend_direction']}. "
        f"Minimum closing rank: {row['min_closing']:,}, "
        f"Maximum closing rank: {row['max_closing']:,}, "
        f"Average closing rank: {row['avg_closing']:,.0f}."
    )

    metadata = {
        'doc_type'       : 'trend_summary',
        'institute'      : row['institute'],
        'program'        : row['program'],
        'category_base'  : row['category_base'],
        'gender_short'   : row['gender_short'],
        'quota'          : row['quota'],
        'avg_closing'    : float(row['avg_closing']),
        'trend_direction': row['trend_direction']
    }

    summary_docs.append(
        Document(page_content=text, metadata=metadata)
    )

print(f"Trend summary documents created: {len(summary_docs):,}")

# Summary 2: College overall summary
college_summary = (
    df_rag.groupby(['institute', 'institute_type'])
    .agg(
        avg_closing=('closing_rank', 'mean'),
        min_closing=('closing_rank', 'min'),
        max_closing=('closing_rank', 'max'),
        total_programs=('program', 'nunique'),
        years_data=('year', 'nunique')
    )
    .reset_index()
    .sort_values('avg_closing')
)

for _, row in college_summary.iterrows():
    text = (
        f"College summary for {row['institute']} "
        f"({row['institute_type']}): "
        f"Average closing rank across all programs and categories "
        f"is {row['avg_closing']:,.0f}. "
        f"Minimum closing rank: {row['min_closing']:,}. "
        f"Maximum closing rank: {row['max_closing']:,}. "
        f"Offers {row['total_programs']} unique programs. "
        f"Data available for {row['years_data']} years."
    )

    metadata = {
        'doc_type'      : 'college_summary',
        'institute'     : row['institute'],
        'institute_type': row['institute_type'],
        'avg_closing'   : float(row['avg_closing'])
    }

    summary_docs.append(
        Document(page_content=text, metadata=metadata)
    )

print(f"Total summary documents: {len(summary_docs):,}")

# Combine all documents
all_documents = documents + summary_docs
print(f"\nTotal documents for indexing: {len(all_documents):,}")

Trend summary documents created: 15,348
Total summary documents: 15,484

Total documents for indexing: 81,112


In [13]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
    # removed show_progress_bar — causes conflict
)

test_embed = embeddings.embed_query("test")
print(f"Embedding model loaded: {len(test_embed)} dimensions")

Loading embedding model...


C:\Users\srith\AppData\Local\Temp\ipykernel_15488\2711462714.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embedding model loaded: 384 dimensions


In [15]:
print("Building vector store...")
print(f"Indexing {len(all_documents):,} documents...")
print("This will take 5-15 minutes. Please wait...\n")

batch_size = 500  # smaller batch to avoid memory issues
vectorstore = None

for i in range(0, len(all_documents), batch_size):
    batch = all_documents[i:i+batch_size]

    try:
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)

        progress = min(i + batch_size, len(all_documents))
        print(f"  Indexed {progress:,} / {len(all_documents):,} documents")

    except Exception as e:
        print(f"  Error at batch {i}: {e}")
        continue

print(f"\nVector store built successfully.")
print(f"Total vectors indexed: {vectorstore.index.ntotal:,}")

Building vector store...
Indexing 81,112 documents...
This will take 5-15 minutes. Please wait...

  Indexed 500 / 81,112 documents
  Indexed 1,000 / 81,112 documents
  Indexed 1,500 / 81,112 documents
  Indexed 2,000 / 81,112 documents
  Indexed 2,500 / 81,112 documents
  Indexed 3,000 / 81,112 documents
  Indexed 3,500 / 81,112 documents
  Indexed 4,000 / 81,112 documents
  Indexed 4,500 / 81,112 documents
  Indexed 5,000 / 81,112 documents
  Indexed 5,500 / 81,112 documents
  Indexed 6,000 / 81,112 documents
  Indexed 6,500 / 81,112 documents
  Indexed 7,000 / 81,112 documents
  Indexed 7,500 / 81,112 documents
  Indexed 8,000 / 81,112 documents
  Indexed 8,500 / 81,112 documents
  Indexed 9,000 / 81,112 documents
  Indexed 9,500 / 81,112 documents
  Indexed 10,000 / 81,112 documents
  Indexed 10,500 / 81,112 documents
  Indexed 11,000 / 81,112 documents
  Indexed 11,500 / 81,112 documents
  Indexed 12,000 / 81,112 documents
  Indexed 12,500 / 81,112 documents
  Indexed 13,000 / 81,

In [17]:
# Save the vector store to disk
# So we never need to rebuild it again

vectorstore_path = os.path.join('..', 'models', 
                                'faiss_vectorstore')
os.makedirs(vectorstore_path, exist_ok=True)

vectorstore.save_local(vectorstore_path)
print(f"Vector store saved to: {vectorstore_path}")

Vector store saved to: ..\models\faiss_vectorstore


In [19]:
# Load saved vector store instead of rebuilding
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Reload embeddings
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Load saved vector store from disk
vectorstore_path = os.path.join('..', 'models', 'faiss_vectorstore')
vectorstore = FAISS.load_local(
    vectorstore_path,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Vector store loaded: {vectorstore.index.ntotal:,} vectors")

Vector store loaded: 81,112 vectors


In [21]:
# Initialize the Groq LLM
# We use llama-3.1-8b-instant — fast and capable
# This is the model that will generate answers

llm = ChatGroq(
    model_name='llama-3.1-8b-instant',
    temperature=0.1,       # Low temperature = more factual, less creative
    max_tokens=1024,       # Enough for detailed answers
    groq_api_key=os.environ.get('GROQ_API_KEY')
)

# Test the LLM
test_response = llm.invoke("Say 'JEE RAG system ready' in one line.")
print(f"LLM test: {test_response.content}")

LLM test: The JEE RAG system is ready.


In [87]:
def get_eligible_colleges(student_rank, category, gender, quota, 
                            program_keywords, institute_types=None, 
                            margin_range=(-1000, 2000), top_n=20):
    """
    Wider default range to show more realistic options.
    margin = closing_rank - student_rank
    """
    data = df_rag.copy()
    
    data = data[
        (data['category_base'] == category) &
        (data['gender_short'] == gender) &
        (data['quota'] == quota)
    ]
    
    if institute_types:
        data = data[data['institute_type'].isin(institute_types)]
    
    if program_keywords:
        pattern = '|'.join(program_keywords)
        data = data[data['program'].str.contains(pattern, case=False, na=False)]
    
    data = data.sort_values('year', ascending=False)
    data = data.drop_duplicates(
        subset=['institute', 'program', 'category_base', 'gender_short', 'quota'],
        keep='first'
    )
    
    data['margin'] = data['closing_rank'] - student_rank
    
    filtered = data[
        (data['margin'] >= margin_range[0]) & 
        (data['margin'] <= margin_range[1])
    ].copy()
    
    def verdict(m):
        if m < -200:
            return '🔴 Out of Reach'
        elif m < 0:
            return '🟡 Borderline (Risky)'
        elif m <= 1000:
            return '✅ Safe'
        else:
            return '🟢 Comfortable'
    
    filtered['verdict'] = filtered['margin'].apply(verdict)
    
    # Sort by margin value (negative=harder first, then ascending positive)
    filtered = filtered.sort_values('margin')
    
    return filtered[[
        'institute', 'institute_type', 'program', 'year',
        'closing_rank', 'margin', 'verdict'
    ]].head(top_n)


print("✅ Cell 11 done — wider margin range")

✅ Cell 11 done — wider margin range


In [89]:
def format_admission_results(student_rank, category, results_df, query_description):
    output = []
    output.append("═" * 50)
    output.append("📊 ADMISSION ANALYSIS")
    output.append("═" * 50)
    output.append(f"🎯 Your Rank: {student_rank:,} | Category: {category}")
    output.append("")
    
    if results_df.empty:
        output.append("❌ No colleges found. Try a different branch/category/quota.")
        output.append("═" * 50)
        return "\n".join(output)
    
    def short(name):
        return (name.replace('Indian Institute of Technology', 'IIT')
                     .replace('National Institute of Technology', 'NIT')
                     .replace('Indian Institute of Information Technology', 'IIIT'))
    
    output.append("✅ REALISTIC OPTIONS (sorted by margin):")
    output.append("─" * 50)
    
    for _, row in results_df.iterrows():
        margin = int(row['margin'])
        margin_str = f"+{margin:,}" if margin >= 0 else f"{margin:,}"
        
        output.append(f"\n🏛️  {short(row['institute'])} ({row['institute_type']})")
        output.append(f"   Program       : {row['program'][:75]}")
        output.append(f"   Closing Rank  : {row['closing_rank']:,} ({int(row['year'])})")
        output.append(f"   Margin        : {margin_str} ranks")
        output.append(f"   Verdict       : {row['verdict']}")
    
    output.append("\n" + "═" * 50)
    output.append("💡 RECOMMENDATION")
    output.append("─" * 50)
    
    safe = results_df[results_df['verdict'].isin(['✅ Safe', '🟢 Comfortable'])]
    risky = results_df[results_df['verdict'] == '🟡 Borderline (Risky)']
    
    if not safe.empty:
        best = safe.iloc[0]
        output.append(f"Top safe pick: {short(best['institute'])} — "
                       f"{best['program'][:50]} (closing {best['closing_rank']:,}, "
                       f"margin +{int(best['margin']):,}).")
    
    if not risky.empty:
        output.append(f"\n{len(risky)} borderline option(s) worth trying as a stretch goal "
                       f"if you're willing to take a small risk.")
    
    output.append(f"\nTotal options shown: {len(results_df)} "
                   f"({len(safe)} safe, {len(risky)} risky, "
                   f"{(results_df['verdict']=='🔴 Out of Reach').sum()} out of reach)")
    output.append("═" * 50)
    
    return "\n".join(output)


print("✅ Cell 12 done")

✅ Cell 12 done


In [91]:
def get_trend(institute_search, program_search, category='OPEN', 
               gender='GN', quota='AI'):
    
    # Map common abbreviations -> full institute name fragments
    institute_map = {
        'iit bombay': 'Technology Bombay',
        'iit delhi': 'Technology Delhi',
        'iit madras': 'Technology Madras',
        'iit kanpur': 'Technology Kanpur',
        'iit kharagpur': 'Technology Kharagpur',
        'iit roorkee': 'Technology Roorkee',
        'iit guwahati': 'Technology Guwahati',
        'iit hyderabad': 'Technology Hyderabad',
        'iit bhu': '(BHU) Varanasi',
        'iit patna': 'Technology Patna',
        'nit surathkal': 'Karnataka, Surathkal',
        'nit trichy': 'Tiruchirappalli',
        'nit warangal': 'Warangal',
    }
    
    program_map = {
        'cse': 'Computer Science', 'ece': 'Electronics and Communication',
        'me': 'Mechanical', 'ce': 'Civil', 'ee': 'Electrical Engineering',
        'it': 'Information Technology',
    }
    
    inst_key = institute_search.lower().strip()
    institute_actual = institute_map.get(inst_key, institute_search)
    program_actual = program_map.get(program_search.lower().strip(), program_search)
    
    data = df_rag[
        (df_rag['institute'].str.contains(institute_actual, case=False, na=False, regex=False)) &
        (df_rag['program'].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag['category_base'] == category) &
        (df_rag['gender_short'] == gender) &
        (df_rag['quota'] == quota)
    ].copy()
    
    if data.empty:
        # Fallback: try without quota restriction
        data = df_rag[
            (df_rag['institute'].str.contains(institute_actual, case=False, na=False, regex=False)) &
            (df_rag['program'].str.contains(program_actual, case=False, na=False, regex=False)) &
            (df_rag['category_base'] == category) &
            (df_rag['gender_short'] == gender)
        ].copy()
    
    if data.empty:
        return None, f"No data found for '{institute_search}' + '{program_search}' ({category}, {gender})."
    
    trend = data.groupby('year')['closing_rank'].median().reset_index().sort_values('year')
    
    institute_name = data['institute'].iloc[0]
    program_name = data['program'].iloc[0]
    
    output = []
    output.append("═" * 50)
    output.append("📈 TREND ANALYSIS")
    output.append("═" * 50)
    output.append(f"Institute : {institute_name}")
    output.append(f"Program   : {program_name[:70]}")
    output.append(f"Category  : {category} | {gender}")
    output.append("")
    output.append("Year-wise Closing Rank:")
    
    for _, row in trend.iterrows():
        output.append(f"  • {int(row['year'])}: {int(row['closing_rank']):,}")
    
    if len(trend) >= 2:
        first_rank, last_rank = trend.iloc[0]['closing_rank'], trend.iloc[-1]['closing_rank']
        first_year, last_year = int(trend.iloc[0]['year']), int(trend.iloc[-1]['year'])
        change_pct = ((last_rank - first_rank) / first_rank) * 100
        direction = "increased (less competitive)" if change_pct > 0 else "decreased (more competitive)"
        output.append("")
        output.append("📊 SUMMARY:")
        output.append(f"From {first_year} to {last_year}, closing rank {direction} "
                       f"by {abs(change_pct):.1f}% ({int(first_rank):,} → {int(last_rank):,}).")
    
    output.append("═" * 50)
    return trend, "\n".join(output)


# Test
_, report = get_trend('iit bombay', 'cse', 'OPEN', 'GN', 'AI')
print(report)

══════════════════════════════════════════════════
📈 TREND ANALYSIS
══════════════════════════════════════════════════
Institute : Indian Institute of Technology Bombay
Program   : Computer Science and Engineering (4 Years, Bachelor of Technology)
Category  : OPEN | GN

Year-wise Closing Rank:
  • 2018: 59
  • 2019: 63
  • 2020: 66
  • 2021: 67
  • 2022: 61
  • 2023: 67
  • 2024: 68
  • 2025: 66

📊 SUMMARY:
From 2018 to 2025, closing rank increased (less competitive) by 11.9% (59 → 66).
══════════════════════════════════════════════════


In [93]:
def get_best_colleges_for_program(program_search, institute_type=None, 
                                    category='OPEN', gender='GN', 
                                    quota=None, top_n=10):
    """
    Find the most competitive (best) colleges for a given program.
    'Best' = lowest median closing rank.
    """
    program_map = {
        'cse': 'Computer Science', 'ece': 'Electronics and Communication',
        'me': 'Mechanical', 'ce': 'Civil', 'ee': 'Electrical Engineering',
        'it': 'Information Technology',
    }
    program_actual = program_map.get(program_search.lower().strip(), program_search)
    
    data = df_rag[
        (df_rag['program'].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag['category_base'] == category) &
        (df_rag['gender_short'] == gender)
    ].copy()
    
    if institute_type:
        data = data[data['institute_type'] == institute_type]
    
    if quota:
        data = data[data['quota'] == quota]
    else:
        # Use OS for NIT, AI for others as default "best comparison" quota
        data = data[data['quota'].isin(['AI', 'OS'])]
    
    if data.empty:
        return f"No data found for '{program_search}'."
    
    # Use most recent 2 years for relevance
    recent_years = sorted(data['year'].unique())[-2:]
    data = data[data['year'].isin(recent_years)]
    
    ranking = (
        data.groupby('institute')['closing_rank']
        .median()
        .reset_index()
        .sort_values('closing_rank')
        .head(top_n)
    )
    
    output = []
    output.append("═" * 50)
    output.append(f"🏆 BEST COLLEGES FOR {program_search.upper()}")
    output.append("═" * 50)
    output.append(f"(Based on {category}/{gender} category, years {recent_years})")
    output.append("")
    
    for i, (_, row) in enumerate(ranking.iterrows(), 1):
        short_name = (row['institute']
                       .replace('Indian Institute of Technology', 'IIT')
                       .replace('National Institute of Technology', 'NIT')
                       .replace('Indian Institute of Information Technology', 'IIIT'))
        output.append(f"{i}. {short_name} — Median Closing Rank: {int(row['closing_rank']):,}")
    
    output.append("═" * 50)
    
    return "\n".join(output)


# Test
print(get_best_colleges_for_program('Mechanical', institute_type='NIT'))

══════════════════════════════════════════════════
🏆 BEST COLLEGES FOR MECHANICAL
══════════════════════════════════════════════════
(Based on OPEN/GN category, years [2024, 2025])

1. NIT, Tiruchirappalli — Median Closing Rank: 8,581
2. NIT Karnataka, Surathkal — Median Closing Rank: 11,429
3. NIT, Warangal — Median Closing Rank: 13,637
4. NIT, Rourkela — Median Closing Rank: 14,143
5. NIT Calicut — Median Closing Rank: 18,749
6. Motilal Nehru NIT Allahabad — Median Closing Rank: 18,750
7. Malaviya NIT Jaipur — Median Closing Rank: 19,886
8. NIT Delhi — Median Closing Rank: 20,757
9. NIT, Kurukshetra — Median Closing Rank: 21,191
10. Visvesvaraya NIT, Nagpur — Median Closing Rank: 21,958
══════════════════════════════════════════════════


In [95]:
def jee_assistant(question):
    classification_prompt = f"""Classify this question and extract parameters as JSON.

Question: {question}

Return ONLY valid JSON in one of these formats:

If asking which colleges student can get with a given rank:
{{"type": "admission", "rank": <number>, "category": "OPEN/OBC-NCL/SC/ST/EWS", 
  "gender": "GN/FO", "quota": "AI/OS/HS", "programs": ["keyword1","keyword2"], 
  "institute_types": ["IIT","NIT","IIIT","GFTI"]}}

If asking about historical trend for ONE specific institute+program:
{{"type": "trend", "institute": "<search term>", "program": "<cse/ece/me/ce/ee/it or full name>",
  "category": "OPEN/OBC-NCL/SC/ST/EWS", "gender": "GN/FO", "quota": "AI/OS/HS"}}

If asking which college/institute is BEST for a program (ranking question):
{{"type": "best_college", "program": "<cse/ece/me/ce/ee/it or full name>", 
  "institute_type": "IIT/NIT/IIIT/GFTI or null", "category": "OPEN/OBC-NCL/SC/ST/EWS",
  "gender": "GN/FO"}}

Otherwise:
{{"type": "general"}}

Default category=OPEN, gender=GN, quota=AI if not specified.
Return ONLY the JSON, no other text."""
    
    response = llm.invoke(classification_prompt)
    
    import json, re
    json_text = response.content.strip()
    json_match = re.search(r'\{.*\}', json_text, re.DOTALL)
    if json_match:
        json_text = json_match.group()
    
    try:
        params = json.loads(json_text)
    except:
        params = {"type": "general"}
    
    print(f"[Agent routing decision: {params}]\n")
    
    if params.get('type') == 'admission':
        results = get_eligible_colleges(
            student_rank=params.get('rank', 5000),
            category=params.get('category', 'OPEN'),
            gender=params.get('gender', 'GN'),
            quota=params.get('quota', 'AI'),
            program_keywords=params.get('programs', []),
            institute_types=params.get('institute_types', None),
            top_n=15
        )
        return format_admission_results(
            student_rank=params.get('rank', 5000),
            category=params.get('category', 'OPEN'),
            results_df=results,
            query_description=question
        )
    
    elif params.get('type') == 'trend':
        _, report = get_trend(
            institute_search=params.get('institute', ''),
            program_search=params.get('program', ''),
            category=params.get('category', 'OPEN'),
            gender=params.get('gender', 'GN'),
            quota=params.get('quota', 'AI')
        )
        return report
    
    elif params.get('type') == 'best_college':
        return get_best_colleges_for_program(
            program_search=params.get('program', ''),
            institute_type=params.get('institute_type'),
            category=params.get('category', 'OPEN'),
            gender=params.get('gender', 'GN')
        )
    
    else:
        return ask_trend_question(question)


print("✅ Cell 15 updated — full agent router with 3 tools")

✅ Cell 15 updated — full agent router with 3 tools


In [97]:
print(jee_assistant(
    "I have rank 2600 OPEN category GN gender AI quota. "
    "I want CSE or Mathematics and Computing at IITs and CSE at NITs."
))

[Agent routing decision: {'type': 'admission', 'rank': 2600, 'category': 'OPEN', 'gender': 'GN', 'quota': 'AI', 'programs': ['CSE', 'Mathematics and Computing'], 'institute_types': ['IIT', 'NIT']}]

══════════════════════════════════════════════════
📊 ADMISSION ANALYSIS
══════════════════════════════════════════════════
🎯 Your Rank: 2,600 | Category: OPEN

✅ REALISTIC OPTIONS (sorted by margin):
──────────────────────────────────────────────────

🏛️  IIT (BHU) Varanasi (IIT)
   Program       : Mathematics and Computing (4 Years, Bachelor of Technology)
   Closing Rank  : 1,850 (2025)
   Margin        : -750 ranks
   Verdict       : 🔴 Out of Reach

🏛️  IIT Indore (IIT)
   Program       : Mathematics and Computing (4 Years, Bachelor of Technology)
   Closing Rank  : 2,035 (2025)
   Margin        : -565 ranks
   Verdict       : 🔴 Out of Reach

🏛️  IIT (BHU) Varanasi (IIT)
   Program       : Mathematics and Computing (5 Years, Bachelor and Master of Technology (Dual
   Closing Rank  : 2,15

In [101]:
print(jee_assistant(
    "How has IIT Madras CSE closing rank changed from 2018 to 2024?"
))

[Agent routing decision: {'type': 'trend', 'institute': 'IIT Madras', 'program': 'CSE', 'category': 'OPEN', 'gender': 'GN', 'quota': 'AI'}]

══════════════════════════════════════════════════
📈 TREND ANALYSIS
══════════════════════════════════════════════════
Institute : Indian Institute of Technology Madras
Program   : Computer Science and Engineering (4 Years, Bachelor of Technology)
Category  : OPEN | GN

Year-wise Closing Rank:
  • 2018: 200
  • 2019: 188
  • 2020: 158
  • 2021: 163
  • 2022: 175
  • 2023: 148
  • 2024: 159
  • 2025: 171

📊 SUMMARY:
From 2018 to 2025, closing rank decreased (more competitive) by 14.5% (200 → 171).
══════════════════════════════════════════════════


In [77]:
print(jee_assistant(
    "Which NIT is best for Mechanical Engineering?"
))

[Agent routing decision: {'type': 'best_college', 'program': 'Mechanical Engineering', 'institute_type': 'NIT', 'category': 'OPEN', 'gender': 'GN'}]

══════════════════════════════════════════════════
🏆 BEST COLLEGES FOR MECHANICAL ENGINEERING
══════════════════════════════════════════════════
(Based on OPEN/GN category, years [2024, 2025])

1. NIT, Tiruchirappalli — Median Closing Rank: 8,581
2. NIT Karnataka, Surathkal — Median Closing Rank: 11,429
3. NIT, Warangal — Median Closing Rank: 13,637
4. NIT, Rourkela — Median Closing Rank: 14,143
5. NIT Calicut — Median Closing Rank: 18,749
6. Motilal Nehru NIT Allahabad — Median Closing Rank: 18,750
7. Malaviya NIT Jaipur — Median Closing Rank: 19,886
8. NIT Delhi — Median Closing Rank: 20,757
9. NIT, Kurukshetra — Median Closing Rank: 21,191
10. Visvesvaraya NIT, Nagpur — Median Closing Rank: 21,958
══════════════════════════════════════════════════


In [103]:
import os
import shutil

# Create deployment folder
deploy_dir = os.path.join('..', 'deployment')
os.makedirs(deploy_dir, exist_ok=True)
os.makedirs(os.path.join(deploy_dir, 'data'), exist_ok=True)
os.makedirs(os.path.join(deploy_dir, 'models'), exist_ok=True)

# Copy required data file
shutil.copy(
    os.path.join('..', 'data', 'processed', 'josaa_featured.csv'),
    os.path.join(deploy_dir, 'data', 'josaa_featured.csv')
)

# Copy FAISS vectorstore
faiss_src = os.path.join('..', 'models', 'faiss_vectorstore')
faiss_dst = os.path.join(deploy_dir, 'models', 'faiss_vectorstore')
if os.path.exists(faiss_dst):
    shutil.rmtree(faiss_dst)
shutil.copytree(faiss_src, faiss_dst)

print("✅ Deployment folder structure created")
print(f"Location: {os.path.abspath(deploy_dir)}")

# Check sizes
for root, dirs, files in os.walk(deploy_dir):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  {path}: {size_mb:.2f} MB")

✅ Deployment folder structure created
Location: C:\Users\srith\jee_project\deployment
  ..\deployment\data\josaa_featured.csv: 109.82 MB
  ..\deployment\models\faiss_vectorstore\index.faiss: 118.82 MB
  ..\deployment\models\faiss_vectorstore\index.pkl: 41.60 MB


In [105]:
requirements = """streamlit==1.32.0
pandas==2.2.0
numpy==1.26.4
scikit-learn==1.4.2
joblib==1.4.2
langchain==1.3.4
langchain-core==1.4.1
langchain-community==0.4.2
langchain-groq==1.1.2
langchain-text-splitters==1.1.2
faiss-cpu==1.14.2
sentence-transformers==5.5.1
transformers==4.44.0
torch==2.2.2
"""

with open(os.path.join(deploy_dir, 'requirements.txt'), 'w') as f:
    f.write(requirements)

print("✅ requirements.txt created")

✅ requirements.txt created


In [107]:
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import os
import json
import re
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# ───────────────────────────────────────────────
# PAGE CONFIG
# ───────────────────────────────────────────────
st.set_page_config(
    page_title="JEE Admission Counselor AI",
    page_icon="🎓",
    layout="wide"
)

# ───────────────────────────────────────────────
# LOAD RESOURCES (cached so it loads once)
# ───────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("data/josaa_featured.csv")
    last_round = (
        df[df["is_special_round"] == False]
        .groupby(["year", "institute_type"])["round"]
        .max()
        .reset_index()
        .rename(columns={"round": "last_round"})
    )
    df_merged = df.merge(last_round, on=["year", "institute_type"], how="left")
    df_rag = df_merged[
        (df_merged["round"] == df_merged["last_round"]) &
        (df_merged["is_special_round"] == False) &
        (df_merged["is_pwd"] == False)
    ].copy()
    return df_rag

@st.cache_resource
def load_embeddings():
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    return HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

@st.cache_resource
def load_vectorstore(_embeddings):
    return FAISS.load_local(
        "models/faiss_vectorstore",
        _embeddings,
        allow_dangerous_deserialization=True
    )

@st.cache_resource
def load_llm():
    api_key = st.secrets.get("GROQ_API_KEY", os.environ.get("GROQ_API_KEY", ""))
    return ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.1,
        max_tokens=1024,
        groq_api_key=api_key
    )

df_rag = load_data()
embeddings = load_embeddings()
vectorstore = load_vectorstore(embeddings)
llm = load_llm()

# ───────────────────────────────────────────────
# CORE TOOLS (same logic as Phase 12/13)
# ───────────────────────────────────────────────
def get_eligible_colleges(student_rank, category, gender, quota,
                            program_keywords, institute_types=None,
                            margin_range=(-2000, 5000), top_n=20):
    data = df_rag.copy()
    data = data[
        (data["category_base"] == category) &
        (data["gender_short"] == gender) &
        (data["quota"] == quota)
    ]
    if institute_types:
        data = data[data["institute_type"].isin(institute_types)]
    if program_keywords:
        pattern = "|".join(program_keywords)
        data = data[data["program"].str.contains(pattern, case=False, na=False)]

    data = data.sort_values("year", ascending=False)
    data = data.drop_duplicates(
        subset=["institute", "program", "category_base", "gender_short", "quota"],
        keep="first"
    )
    data["margin"] = data["closing_rank"] - student_rank
    filtered = data[
        (data["margin"] >= margin_range[0]) &
        (data["margin"] <= margin_range[1])
    ].copy()

    def verdict(m):
        if m < -200:
            return "🔴 Out of Reach"
        elif m < 0:
            return "🟡 Borderline (Risky)"
        elif m <= 1000:
            return "✅ Safe"
        else:
            return "🟢 Comfortable"

    filtered["verdict"] = filtered["margin"].apply(verdict)
    filtered = filtered.sort_values("margin")
    return filtered[[
        "institute", "institute_type", "program", "year",
        "closing_rank", "margin", "verdict"
    ]].head(top_n)


def short_name(name):
    return (name.replace("Indian Institute of Technology", "IIT")
                 .replace("National Institute of Technology", "NIT")
                 .replace("Indian Institute of Information Technology", "IIIT"))


def get_trend(institute_search, program_search, category="OPEN",
               gender="GN", quota="AI"):
    institute_map = {
        "iit bombay": "Technology Bombay", "iit delhi": "Technology Delhi",
        "iit madras": "Technology Madras", "iit kanpur": "Technology Kanpur",
        "iit kharagpur": "Technology Kharagpur", "iit roorkee": "Technology Roorkee",
        "iit guwahati": "Technology Guwahati", "iit hyderabad": "Technology Hyderabad",
        "iit bhu": "(BHU) Varanasi", "iit patna": "Technology Patna",
        "nit surathkal": "Karnataka, Surathkal", "nit trichy": "Tiruchirappalli",
        "nit warangal": "Warangal",
    }
    program_map = {
        "cse": "Computer Science", "ece": "Electronics and Communication",
        "me": "Mechanical", "ce": "Civil", "ee": "Electrical Engineering",
        "it": "Information Technology",
    }
    inst_key = institute_search.lower().strip()
    institute_actual = institute_map.get(inst_key, institute_search)
    program_actual = program_map.get(program_search.lower().strip(), program_search)

    data = df_rag[
        (df_rag["institute"].str.contains(institute_actual, case=False, na=False, regex=False)) &
        (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag["category_base"] == category) &
        (df_rag["gender_short"] == gender) &
        (df_rag["quota"] == quota)
    ].copy()

    if data.empty:
        data = df_rag[
            (df_rag["institute"].str.contains(institute_actual, case=False, na=False, regex=False)) &
            (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
            (df_rag["category_base"] == category) &
            (df_rag["gender_short"] == gender)
        ].copy()

    if data.empty:
        return None

    trend = data.groupby("year")["closing_rank"].median().reset_index().sort_values("year")
    trend["institute_name"] = data["institute"].iloc[0]
    trend["program_name"] = data["program"].iloc[0]
    return trend


def get_best_colleges_for_program(program_search, institute_type=None,
                                    category="OPEN", gender="GN", quota=None, top_n=10):
    program_map = {
        "cse": "Computer Science", "ece": "Electronics and Communication",
        "me": "Mechanical", "ce": "Civil", "ee": "Electrical Engineering",
        "it": "Information Technology",
    }
    program_actual = program_map.get(program_search.lower().strip(), program_search)

    data = df_rag[
        (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag["category_base"] == category) &
        (df_rag["gender_short"] == gender)
    ].copy()

    if institute_type:
        data = data[data["institute_type"] == institute_type]
    if quota:
        data = data[data["quota"] == quota]
    else:
        data = data[data["quota"].isin(["AI", "OS"])]

    if data.empty:
        return None

    recent_years = sorted(data["year"].unique())[-2:]
    data = data[data["year"].isin(recent_years)]

    ranking = (
        data.groupby("institute")["closing_rank"]
        .median().reset_index().sort_values("closing_rank").head(top_n)
    )
    ranking["recent_years"] = str(recent_years)
    return ranking


def ask_general_question(question):
    docs = vectorstore.similarity_search(question, k=15)
    context = "\\\\n\\\\n".join(d.page_content for d in docs[:15])
    prompt = f"""Based on this JoSAA admission data, answer the question with 
specific year-wise numbers only. Be concise.

DATA:
{context}

QUESTION: {question}

ANSWER:"""
    response = llm.invoke(prompt)
    return response.content


def classify_question(question):
    classification_prompt = f"""Classify this question and extract parameters as JSON.

Question: {question}

Return ONLY valid JSON in one of these formats:

If asking which colleges student can get with a given rank:
{{{{"type": "admission", "rank": <number>, "category": "OPEN/OBC-NCL/SC/ST/EWS",
  "gender": "GN/FO", "quota": "AI/OS/HS", "programs": ["keyword1","keyword2"],
  "institute_types": ["IIT","NIT","IIIT","GFTI"]}}}}

If asking about historical trend for ONE specific institute+program:
{{{{"type": "trend", "institute": "<search term>", "program": "<cse/ece/me/ce/ee/it or full name>",
  "category": "OPEN/OBC-NCL/SC/ST/EWS", "gender": "GN/FO", "quota": "AI/OS/HS"}}}}

If asking which college is BEST for a program:
{{{{"type": "best_college", "program": "<cse/ece/me/ce/ee/it or full name>",
  "institute_type": "IIT/NIT/IIIT/GFTI or null", "category": "OPEN/OBC-NCL/SC/ST/EWS",
  "gender": "GN/FO"}}}}

Otherwise:
{{{{"type": "general"}}}}

Default category=OPEN, gender=GN, quota=AI if not specified.
Return ONLY the JSON, no other text."""

    response = llm.invoke(classification_prompt)
    json_text = response.content.strip()
    json_match = re.search(r"\\\\{.*\\\\}", json_text, re.DOTALL)
    if json_match:
        json_text = json_match.group()
    try:
        return json.loads(json_text)
    except:
        return {"type": "general"}


# ───────────────────────────────────────────────
# UI
# ───────────────────────────────────────────────
st.title("🎓 JEE Admission Counselor AI")
st.caption("Powered by Machine Learning + RAG + LLaMA 3.1 | Built on 8 years of JoSAA data (2018-2025)")

with st.sidebar:
    st.header("ℹ️ About")
    st.markdown("""
    This AI counselor uses:
    - **432,524** JoSAA admission records
    - **Random Forest ML model** (R²=0.87)
    - **FAISS vector search** over 81,112 documents
    - **Groq LLaMA-3.1** for natural language understanding

    **Example questions:**
    - "I have rank 5000 OPEN category, GN, AI quota. Which NITs can I get for CSE?"
    - "How has IIT Delhi CSE closing rank changed from 2018 to 2024?"
    - "Which NIT is best for Mechanical Engineering?"
    """)

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Hi! I'\\''m your JEE Admission Counselor. Ask me about cutoffs, trends, or which colleges you can get with your rank."}
    ]

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if question := st.chat_input("Ask about JEE admissions..."):
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("Analyzing..."):
            params = classify_question(question)

            if params.get("type") == "admission":
                results = get_eligible_colleges(
                    student_rank=params.get("rank", 5000),
                    category=params.get("category", "OPEN"),
                    gender=params.get("gender", "GN"),
                    quota=params.get("quota", "AI"),
                    program_keywords=params.get("programs", []),
                    institute_types=params.get("institute_types", None),
                )
                if results.empty:
                    response = "❌ No colleges found matching your criteria. Try a different branch, category, or quota."
                else:
                    response = f"### 📊 Results for Rank {params.get('rank'):,} ({params.get('category')})\\\\n\\\\n"
                    for _, row in results.iterrows():
                        margin = int(row["margin"])
                        margin_str = f"+{margin:,}" if margin >= 0 else f"{margin:,}"
                        response += f"**{short_name(row['institute'])}** ({row['institute_type']})  \\\\n"
                        response += f"{row['program'][:80]}  \\\\n"
                        response += f"Closing Rank: {row['closing_rank']:,} ({int(row['year'])}) | Margin: {margin_str} | {row['verdict']}\\\\n\\\\n"

            elif params.get("type") == "trend":
                trend = get_trend(
                    institute_search=params.get("institute", ""),
                    program_search=params.get("program", ""),
                    category=params.get("category", "OPEN"),
                    gender=params.get("gender", "GN"),
                    quota=params.get("quota", "AI"),
                )
                if trend is None:
                    response = "❌ No data found for that institute/program combination."
                else:
                    response = f"### 📈 Trend: {trend['institute_name'].iloc[0]}\\\\n\\\\n"
                    response += f"**Program:** {trend['program_name'].iloc[0][:80]}\\\\n\\\\n"
                    for _, row in trend.iterrows():
                        response += f"- **{int(row['year'])}**: {int(row['closing_rank']):,}\\\\n"
                    if len(trend) >= 2:
                        first, last = trend.iloc[0]["closing_rank"], trend.iloc[-1]["closing_rank"]
                        pct = ((last - first) / first) * 100
                        direction = "increased" if pct > 0 else "decreased"
                        response += f"\\\\n**Summary:** Closing rank {direction} by {abs(pct):.1f}% over this period."

            elif params.get("type") == "best_college":
                ranking = get_best_colleges_for_program(
                    program_search=params.get("program", ""),
                    institute_type=params.get("institute_type"),
                    category=params.get("category", "OPEN"),
                    gender=params.get("gender", "GN"),
                )
                if ranking is None:
                    response = "❌ No data found for that program."
                else:
                    response = f"### 🏆 Best Colleges for {params.get('program', '').upper()}\\\\n\\\\n"
                    for i, (_, row) in enumerate(ranking.iterrows(), 1):
                        response += f"{i}. **{short_name(row['institute'])}** — Median Closing Rank: {int(row['closing_rank']):,}\\\\n"

            else:
                response = ask_general_question(question)

        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

with open(os.path.join(deploy_dir, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ app.py created")
print(f"Size: {os.path.getsize(os.path.join(deploy_dir, 'app.py'))} bytes")

✅ app.py created
Size: 15184 bytes


In [109]:
# Load full featured data
df_full = pd.read_csv(os.path.join('..', 'data', 'processed', 'josaa_featured.csv'))

# Keep only columns the app actually uses
needed_cols = [
    'institute', 'institute_type', 'program', 'category_base',
    'gender_short', 'quota', 'year', 'round', 'is_special_round',
    'is_pwd', 'opening_rank', 'closing_rank'
]

df_trimmed = df_full[needed_cols].copy()

# Save trimmed version
trimmed_path = os.path.join(deploy_dir, 'data', 'josaa_featured.csv')
df_trimmed.to_csv(trimmed_path, index=False)

old_size = 109.82
new_size = os.path.getsize(trimmed_path) / (1024*1024)

print(f"✅ Trimmed data file saved")
print(f"Old size: {old_size:.2f} MB")
print(f"New size: {new_size:.2f} MB")
print(f"Reduction: {(1 - new_size/old_size)*100:.1f}%")
print(f"\nColumns kept: {needed_cols}")
print(f"Rows: {len(df_trimmed):,}")

✅ Trimmed data file saved
Old size: 109.82 MB
New size: 66.20 MB
Reduction: 39.7%

Columns kept: ['institute', 'institute_type', 'program', 'category_base', 'gender_short', 'quota', 'year', 'round', 'is_special_round', 'is_pwd', 'opening_rank', 'closing_rank']
Rows: 432,524


In [111]:
readme_content = """---
title: JEE Admission Counselor AI
emoji: 🎓
colorFrom: blue
colorTo: green
sdk: streamlit
sdk_version: 1.32.0
app_file: app.py
pinned: false
---

# 🎓 JEE Admission Counselor AI

An AI-powered admission counseling assistant built on 8 years (2018-2025) of 
JoSAA (Joint Seat Allocation Authority) cutoff data covering 432,524 admission 
records across 136 institutes (IITs, NITs, IIITs, GFTIs).

## Features

- **Rank-based College Finder**: Enter your JEE rank, category, and preferences 
  to get realistic college recommendations with safety verdicts
- **Trend Analysis**: Track how closing ranks for specific colleges/programs 
  changed over the years
- **Best College Rankings**: Find the most competitive colleges for any branch
- **Natural Language Interface**: Powered by Groq LLaMA-3.1 for intent understanding

## Tech Stack

- **Data Processing**: Pandas, NumPy
- **Machine Learning**: Random Forest Regressor (R² = 0.87) for closing rank prediction
- **RAG**: FAISS vector store with 81,112 indexed documents, 
  HuggingFace sentence-transformers embeddings
- **LLM**: Groq (LLaMA-3.1-8b-instant) for query understanding
- **UI**: Streamlit

## Architecture

The system uses a hybrid approach:
1. LLM classifies user intent and extracts structured parameters
2. Deterministic pandas-based engines compute exact rankings/filters/trends
3. RAG handles open-ended comparison questions

This avoids LLM hallucination on numerical data while retaining natural 
language flexibility.

## Data Source

JoSAA Cutoff data 2018-2025 (Rounds 1-6), sourced from official JoSAA 
counselling records via Kaggle.
"""

with open(os.path.join(deploy_dir, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("✅ README.md created for Hugging Face Space")

✅ README.md created for Hugging Face Space


In [3]:
import os
import shutil
import pandas as pd

# Just redefine the path — folder already exists with files
deploy_dir = os.path.join('..', 'deployment')

print(f"✅ Variables reloaded")
print(f"deploy_dir: {os.path.abspath(deploy_dir)}")
print(f"Folder exists: {os.path.exists(deploy_dir)}")

✅ Variables reloaded
deploy_dir: C:\Users\srith\jee_project\deployment
Folder exists: True


In [11]:
import os

deploy_dir = os.path.join('..', 'deployment')

app_code = """import streamlit as st
import pandas as pd
import numpy as np
import os
import json
import re
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

st.set_page_config(page_title="JEE Admission Counselor AI", page_icon="🎓", layout="wide")

@st.cache_data
def load_data():
    df = pd.read_csv("data/josaa_featured.csv")
    last_round = (
        df[df["is_special_round"] == False]
        .groupby(["year", "institute_type"])["round"]
        .max().reset_index().rename(columns={"round": "last_round"})
    )
    df_merged = df.merge(last_round, on=["year", "institute_type"], how="left")
    df_rag = df_merged[
        (df_merged["round"] == df_merged["last_round"]) &
        (df_merged["is_special_round"] == False) &
        (df_merged["is_pwd"] == False)
    ].copy()
    return df_rag

@st.cache_resource
def load_embeddings():
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    return HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

@st.cache_resource
def load_vectorstore(_embeddings):
    return FAISS.load_local(
        "models/faiss_vectorstore",
        _embeddings,
        allow_dangerous_deserialization=True
    )

@st.cache_resource
def load_llm():
    api_key = st.secrets.get("GROQ_API_KEY", os.environ.get("GROQ_API_KEY", ""))
    return ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0.1,
        max_tokens=1024,
        groq_api_key=api_key
    )

df_rag    = load_data()
embeddings = load_embeddings()
vectorstore = load_vectorstore(embeddings)
llm       = load_llm()

def get_eligible_colleges(student_rank, category, gender, quota,
                           program_keywords, institute_types=None,
                           margin_range=(-2000, 5000), top_n=20):
    data = df_rag.copy()
    data = data[
        (data["category_base"] == category) &
        (data["gender_short"] == gender) &
        (data["quota"] == quota)
    ]
    if institute_types:
        data = data[data["institute_type"].isin(institute_types)]
    if program_keywords:
        pattern = "|".join(program_keywords)
        data = data[data["program"].str.contains(pattern, case=False, na=False)]
    data = data.sort_values("year", ascending=False)
    data = data.drop_duplicates(
        subset=["institute", "program", "category_base", "gender_short", "quota"],
        keep="first"
    )
    data["margin"] = data["closing_rank"] - student_rank
    filtered = data[
        (data["margin"] >= margin_range[0]) &
        (data["margin"] <= margin_range[1])
    ].copy()

    def verdict(m):
        if m < -200:   return "Out of Reach"
        elif m < 0:    return "Borderline (Risky)"
        elif m <= 1000: return "Safe"
        else:          return "Comfortable"

    filtered["verdict"] = filtered["margin"].apply(verdict)
    filtered = filtered.sort_values("margin")
    return filtered[[
        "institute", "institute_type", "program",
        "year", "closing_rank", "margin", "verdict"
    ]].head(top_n)


def short_name(name):
    return (name
            .replace("Indian Institute of Technology", "IIT")
            .replace("National Institute of Technology", "NIT")
            .replace("Indian Institute of Information Technology", "IIIT"))


def get_trend(institute_search, program_search,
               category="OPEN", gender="GN", quota="AI"):
    institute_map = {
        "iit bombay"   : "Technology Bombay",
        "iit delhi"    : "Technology Delhi",
        "iit madras"   : "Technology Madras",
        "iit kanpur"   : "Technology Kanpur",
        "iit kharagpur": "Technology Kharagpur",
        "iit roorkee"  : "Technology Roorkee",
        "iit guwahati" : "Technology Guwahati",
        "iit hyderabad": "Technology Hyderabad",
        "iit bhu"      : "(BHU) Varanasi",
        "iit patna"    : "Technology Patna",
        "nit surathkal": "Karnataka, Surathkal",
        "nit trichy"   : "Tiruchirappalli",
        "nit warangal" : "Warangal",
    }
    program_map = {
        "cse": "Computer Science",
        "ece": "Electronics and Communication",
        "me" : "Mechanical",
        "ce" : "Civil",
        "ee" : "Electrical Engineering",
        "it" : "Information Technology",
    }
    institute_actual = institute_map.get(institute_search.lower().strip(), institute_search)
    program_actual   = program_map.get(program_search.lower().strip(), program_search)

    data = df_rag[
        (df_rag["institute"].str.contains(institute_actual, case=False, na=False, regex=False)) &
        (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag["category_base"] == category) &
        (df_rag["gender_short"] == gender) &
        (df_rag["quota"] == quota)
    ].copy()

    if data.empty:
        data = df_rag[
            (df_rag["institute"].str.contains(institute_actual, case=False, na=False, regex=False)) &
            (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
            (df_rag["category_base"] == category) &
            (df_rag["gender_short"] == gender)
        ].copy()

    if data.empty:
        return None

    trend = (
        data.groupby("year")["closing_rank"]
        .median().reset_index().sort_values("year")
    )
    trend["institute_name"] = data["institute"].iloc[0]
    trend["program_name"]   = data["program"].iloc[0]
    return trend


def get_best_colleges(program_search, institute_type=None,
                       category="OPEN", gender="GN", quota=None, top_n=10):
    program_map = {
        "cse": "Computer Science",
        "ece": "Electronics and Communication",
        "me" : "Mechanical",
        "ce" : "Civil",
        "ee" : "Electrical Engineering",
        "it" : "Information Technology",
    }
    program_actual = program_map.get(program_search.lower().strip(), program_search)
    data = df_rag[
        (df_rag["program"].str.contains(program_actual, case=False, na=False, regex=False)) &
        (df_rag["category_base"] == category) &
        (df_rag["gender_short"] == gender)
    ].copy()
    if institute_type:
        data = data[data["institute_type"] == institute_type]
    if quota:
        data = data[data["quota"] == quota]
    else:
        data = data[data["quota"].isin(["AI", "OS"])]
    if data.empty:
        return None
    recent = sorted(data["year"].unique())[-2:]
    data   = data[data["year"].isin(recent)]
    return (
        data.groupby("institute")["closing_rank"]
        .median().reset_index()
        .sort_values("closing_rank").head(top_n)
    )


def ask_general(question):
    docs    = vectorstore.similarity_search(question, k=15)
    context = "\\n\\n".join(d.page_content for d in docs[:15])
    prompt  = (
        "Based on this JoSAA admission data, answer concisely "
        "with specific year-wise numbers.\\n\\n"
        f"DATA:\\n{context}\\n\\nQUESTION: {question}\\n\\nANSWER:"
    )
    return llm.invoke(prompt).content


def classify(question):
    prompt = f\"\"\"Classify this question and return ONLY valid JSON.

Question: {question}

Formats:
Admission: {{"type":"admission","rank":<int>,"category":"OPEN/OBC-NCL/SC/ST/EWS","gender":"GN/FO","quota":"AI/OS/HS","programs":["keyword"],"institute_types":["IIT/NIT/IIIT/GFTI"]}}
Trend:     {{"type":"trend","institute":"<name>","program":"<cse/ece/me/ce/ee/it>","category":"OPEN","gender":"GN","quota":"AI"}}
Best:      {{"type":"best_college","program":"<name>","institute_type":"IIT/NIT/IIIT/GFTI or null","category":"OPEN","gender":"GN"}}
General:   {{"type":"general"}}

Defaults: category=OPEN, gender=GN, quota=AI.
Return ONLY the JSON.\"\"\"
    resp = llm.invoke(prompt).content.strip()
    m    = re.search(r"\\{.*\\}", resp, re.DOTALL)
    if m:
        resp = m.group()
    try:
        return json.loads(resp)
    except:
        return {"type": "general"}


# ── UI ──────────────────────────────────────────────────────
st.title("JEE Admission Counselor AI")
st.caption("Powered by ML + RAG + LLaMA 3.1 | 8 years of JoSAA data (2018-2025)")

with st.sidebar:
    st.header("About")
    st.markdown(\"\"\"
This AI counselor uses:
- **432,524** JoSAA admission records (2018-2025)
- **Random Forest** model (R2 = 0.87)
- **FAISS** vector search over 81,112 documents
- **Groq LLaMA-3.1** for natural language understanding

**Example questions:**
- I have rank 5000 OPEN GN AI quota. Which NITs for CSE?
- How has IIT Delhi CSE closing rank changed 2018 to 2024?
- Which NIT is best for Mechanical Engineering?
\"\"\")

if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "assistant",
        "content": "Hi! I am your JEE Admission Counselor. Ask me about cutoffs, trends, or which colleges you can get with your rank."
    }]

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if question := st.chat_input("Ask about JEE admissions..."):
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("Analyzing..."):
            params = classify(question)

            if params.get("type") == "admission":
                results = get_eligible_colleges(
                    student_rank    = params.get("rank", 5000),
                    category        = params.get("category", "OPEN"),
                    gender          = params.get("gender", "GN"),
                    quota           = params.get("quota", "AI"),
                    program_keywords= params.get("programs", []),
                    institute_types = params.get("institute_types"),
                )
                if results.empty:
                    response = "No colleges found matching your criteria. Try a different branch, category, or quota."
                else:
                    response = f"### Results for Rank {params.get('rank'):,} ({params.get('category')})\\n\\n"
                    for _, row in results.iterrows():
                        m      = int(row["margin"])
                        m_str  = f"+{m:,}" if m >= 0 else f"{m:,}"
                        response += f"**{short_name(row['institute'])}** ({row['institute_type']})  \\n"
                        response += f"{row['program'][:80]}  \\n"
                        response += f"Closing Rank: {row['closing_rank']:,} ({int(row['year'])}) | Margin: {m_str} | {row['verdict']}\\n\\n"

            elif params.get("type") == "trend":
                trend = get_trend(
                    institute_search = params.get("institute", ""),
                    program_search   = params.get("program", ""),
                    category         = params.get("category", "OPEN"),
                    gender           = params.get("gender", "GN"),
                    quota            = params.get("quota", "AI"),
                )
                if trend is None:
                    response = "No data found for that institute and program combination."
                else:
                    response  = f"### Trend: {trend['institute_name'].iloc[0]}\\n\\n"
                    response += f"**Program:** {trend['program_name'].iloc[0][:80]}\\n\\n"
                    for _, row in trend.iterrows():
                        response += f"- **{int(row['year'])}**: {int(row['closing_rank']):,}\\n"
                    if len(trend) >= 2:
                        first = trend.iloc[0]["closing_rank"]
                        last  = trend.iloc[-1]["closing_rank"]
                        pct   = ((last - first) / first) * 100
                        direction = "increased" if pct > 0 else "decreased"
                        response += f"\\n**Summary:** Closing rank {direction} by {abs(pct):.1f}% over this period."

            elif params.get("type") == "best_college":
                ranking = get_best_colleges(
                    program_search = params.get("program", ""),
                    institute_type = params.get("institute_type"),
                    category       = params.get("category", "OPEN"),
                    gender         = params.get("gender", "GN"),
                )
                if ranking is None:
                    response = "No data found for that program."
                else:
                    response = f"### Best Colleges for {params.get('program','').upper()}\\n\\n"
                    for i, (_, row) in enumerate(ranking.iterrows(), 1):
                        response += f"{i}. **{short_name(row['institute'])}** - Median Closing Rank: {int(row['closing_rank']):,}\\n"

            else:
                response = ask_general(question)

        st.markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})
"""

with open(os.path.join(deploy_dir, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_code)

size_kb = os.path.getsize(os.path.join(deploy_dir, 'app.py')) / 1024
print(f"✅ app.py created — {size_kb:.1f} KB")

✅ app.py created — 13.1 KB


In [15]:
print("DEPLOYMENT FOLDER CONTENTS:\n")
for root, dirs, files in os.walk(deploy_dir):
    level = root.replace(deploy_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{indent}  {f} ({size_mb:.2f} MB)")

DEPLOYMENT FOLDER CONTENTS:

deployment/
  app.py (0.01 MB)
  README.md (0.00 MB)
  requirements.txt (0.00 MB)
  data/
    josaa_featured.csv (66.20 MB)
  models/
    faiss_vectorstore/
      index.faiss (118.82 MB)
      index.pkl (41.60 MB)


In [17]:
with open(os.path.join(deploy_dir, 'app.py'), 'r') as f:
    content = f.read()
    
print(f"app.py size: {len(content)} characters")
print(f"\nFirst 200 characters:")
print(content[:200])
print(f"\nLast 100 characters:")
print(content[-100:])

app.py size: 13123 characters

First 200 characters:
import streamlit as st
import pandas as pd
import numpy as np
import os
import json
import re
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from

Last 100 characters:
markdown(response)
    st.session_state.messages.append({"role": "assistant", "content": response})

